# Deep Q-Networks (DQN)

_Modern Deep Learning & AI_

**Let a neural net guess the value of each move. Train it to match reward-now plus the best value next.**

Q-learning keeps a value $Q(s,a)$ for every state-action pair. But real games have far too many states to list.

     A Deep Q-Network replaces that giant table with a neural network. The net reads a state and outputs a $Q$ value for each action.

---

This notebook builds a Deep Q-Network update from scratch, one step at a time. Run each cell, read the note above it, and watch how a single gradient step teaches the net to match reward-now plus the discounted best value next. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We'll perform a single DQN training step on a synthetic mini-batch of transitions. We build it in three parts: (1) the Q-network and its slow target copy, (2) a batch of experience to learn from, (3) the temporal-difference loss and one gradient step.

### Step 1 — Build the Q-network and a target copy

`QNet` is a small two-layer network: it reads a state vector and outputs one $Q$ value per action. We make **two** copies. `q_net` is the network we train; `target_net` is a frozen, slow-moving copy used to compute the learning target. Keeping the target fixed for a while stops the network from chasing its own constantly-shifting predictions, which stabilises training.

In [ ]:
import torch
import torch.nn as nn

class QNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions),   # one Q value per action
        )

    def forward(self, s):
        return self.net(s)

state_dim = 4
n_actions = 2
gamma = 0.9   # discount factor on future reward

q_net = QNet(state_dim, n_actions)
target_net = QNet(state_dim, n_actions)
target_net.load_state_dict(q_net.state_dict())   # target starts as an exact, slow copy

opt = torch.optim.Adam(q_net.parameters(), lr=1e-3)

### Step 2 — Make a mini-batch of transitions

A DQN learns from **transitions**: each is a tuple $(s, a, r, s', \text{done})$ — the state we were in, the action we took, the reward we got, the next state, and whether the episode ended. Here we fake a batch of 8 random transitions so the update has something to chew on; in a real agent these would be sampled from a replay buffer of past experience.

In [ ]:
# A synthetic mini-batch of 8 transitions (s, a, r, s', done).
s = torch.randn(8, state_dim)                 # current states
a = torch.randint(0, n_actions, (8, 1))       # action taken in each
r = torch.randn(8)                            # reward received
s2 = torch.randn(8, state_dim)                # resulting next states
done = torch.zeros(8)                         # 0 = episode still going

### Step 3 — Compute the TD target and take one gradient step

The Bellman target says $Q(s,a)$ should equal reward-now plus the discounted best value available next: $y = r + \gamma \max_{a'} Q_\text{target}(s')$. We read off the network's current estimate $Q(s,a)$ for the action actually taken, build the target with the **frozen** target net (under `no_grad`, so no gradients flow through it), and minimise the squared difference. One optimiser step nudges `q_net` toward the target. The `(1 - done)` factor zeroes out the future term when an episode has ended.

In [ ]:
q_sa = q_net(s).gather(1, a).squeeze(1)        # Q(s, a) for the action taken

with torch.no_grad():
    max_q2 = target_net(s2).max(dim=1).values  # max_a' Q_target(s')
    y = r + gamma * (1 - done) * max_q2        # TD target

loss = nn.functional.mse_loss(q_sa, y)         # (y - Q(s,a))^2

opt.zero_grad()
loss.backward()
opt.step()

print(float(loss))

## Visualize the data & results

_On an 8-state corridor task, is the Q-learning agent actually learning? Does episode reward rise?_

### Step 1 — Run tabular Q-learning on a corridor

To see learning happen end-to-end we drop the neural net and use a plain **table** of $Q$ values on a tiny world: a corridor of 8 states with the goal at the right end. Each episode the agent starts at the left and uses an **epsilon-greedy** policy — mostly taking its best guess, sometimes exploring at random — with epsilon decaying over time. After every move it applies the same Bellman update we used above, written for a table: $Q(s,a) \mathrel{+}= \alpha\,(r + \gamma \max Q(s') - Q(s,a))$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
N = 8         # corridor of 8 states
GOAL = 7      # goal at the right end
Q = np.zeros((N, 2))   # tabular Q: 2 actions (left=0, right=1)
gamma = 0.95
alpha = 0.2
rewards = []

for ep in range(300):
    s = 0
    total = 0.0
    eps = max(0.05, 1.0 - ep / 150)            # epsilon decays as we learn
    for _ in range(30):
        greedy = rng.random() >= eps           # explore vs exploit
        if greedy:
            a = int(np.argmax(Q[s]))           # take current best action
        else:
            a = rng.integers(2)                # random exploratory action

        if a == 1:
            s2 = min(N - 1, s + 1)             # move right
        else:
            s2 = max(0, s - 1)                 # move left

        r = 10.0 if s2 == GOAL else -0.1       # +10 at goal, small step cost
        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])   # Bellman update

        s = s2
        total += r
        if s2 == GOAL:
            break
    rewards.append(total)

### Step 2 — Plot the smoothed learning curve

Raw episode rewards are noisy, so we smooth them with a 20-episode moving average before plotting. If learning is working the curve should climb and then flatten near the maximum reward as the agent reliably walks straight to the goal.

In [ ]:
sm = np.convolve(rewards, np.ones(20) / 20, mode='valid')   # 20-episode moving average

plt.figure(figsize=(6, 4))
plt.plot(sm, color='#4ea1ff', label='episode reward')
plt.xlabel('episode')
plt.ylabel('smoothed episode reward')
plt.title('Tabular Q-learning on 8-state corridor')
plt.legend()
plt.tight_layout()
plt.show()